In [ ]:
import os
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from ipywidgets import interact, IntSlider, FloatSlider


def load_image():
    uploaded = files.upload()
    name = next(iter(uploaded))
    data = np.frombuffer(uploaded[name], np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def show(items, cols=2, figsize=(12, 10)):
    rows = (len(items) + cols - 1) // cols
    plt.figure(figsize=figsize)
    for i, (title, img) in enumerate(items, 1):
        plt.subplot(rows, cols, i)
        if img.ndim == 2:
            plt.imshow(img, cmap='gray', vmin=0, vmax=255)
        else:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()


def background_model(gray, fit_size=96):
    h, w = gray.shape
    small = cv2.resize(gray, (fit_size, fit_size), interpolation=cv2.INTER_AREA).astype(np.float64)
    ys, xs = np.mgrid[0:fit_size, 0:fit_size]
    X = xs.ravel() / fit_size
    Y = ys.ravel() / fit_size
    A = np.stack([np.ones_like(X), X, Y, X ** 2, Y ** 2, X * Y,
                  X ** 3, Y ** 3, X ** 2 * Y, X * Y ** 2], 1)
    z = small.ravel()
    mode = np.bincount(small.astype(np.uint8).ravel(), minlength=256).argmax()
    keep = np.abs(z - mode) < 25
    for _ in range(8):
        coef = np.linalg.lstsq(A[keep], z[keep], rcond=None)[0]
        res = z - A @ coef
        s = max(res[keep].std(), 1.0)
        keep = np.abs(res) < 2 * s
    Xf = (np.arange(w) / w)[None, :]
    Yf = (np.arange(h) / h)[:, None]
    return (coef[0] + coef[1] * Xf + coef[2] * Yf + coef[3] * Xf ** 2 + coef[4] * Yf ** 2
            + coef[5] * Xf * Yf + coef[6] * Xf ** 3 + coef[7] * Yf ** 3
            + coef[8] * Xf ** 2 * Yf + coef[9] * Xf * Yf ** 2)


def binarize(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    bg = background_model(blur)
    diff = np.clip(np.abs(blur.astype(np.float64) - bg), 0, 255).astype(np.uint8)
    _, th = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)
    return th


def split_touching(image, th):
    n, labels = cv2.connectedComponents(th)
    dist = cv2.distanceTransform(th, cv2.DIST_L2, 5)
    sure_fg = np.zeros_like(th)
    for i in range(1, n):
        comp = labels == i
        cnt, _ = cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        c = max(cnt, key=cv2.contourArea)
        hull_area = cv2.contourArea(cv2.convexHull(c))
        solidity = cv2.contourArea(c) / hull_area if hull_area > 0 else 1.0
        ratio = 0.3 if solidity >= 0.93 else 0.75
        sure_fg[comp & (dist > ratio * dist[comp].max())] = 255
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    unknown = cv2.subtract(cv2.dilate(th, np.ones((3, 3), np.uint8)), sure_fg)
    markers[unknown == 255] = 0
    markers = cv2.watershed(image, markers.astype(np.int32))
    lines = cv2.dilate((markers == -1).astype(np.uint8) * 255, np.ones((3, 3), np.uint8))
    out = th.copy()
    out[lines > 0] = 0
    return out


def binarize_uneven(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    k = max(31, (min(gray.shape) // 3) | 1)
    background = cv2.medianBlur(gray, k)
    diff = cv2.absdiff(blur, background)
    _, th = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, kernel)
    return th


In [ ]:
img = load_image()
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray_blur = cv2.GaussianBlur(gray, (5, 5), 0)

sobel_x = cv2.Sobel(gray_blur, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray_blur, cv2.CV_64F, 0, 1, ksize=3)
sobel = cv2.convertScaleAbs(cv2.magnitude(sobel_x, sobel_y))

laplacian = cv2.convertScaleAbs(cv2.Laplacian(gray_blur, cv2.CV_64F, ksize=3))

canny = cv2.Canny(gray_blur, 100, 200)

show([('Исходное изображение', img),
      ('Оператор Собеля', sobel),
      ('Оператор Лапласа', laplacian),
      ('Детектор Кэнни', canny)])

In [ ]:
def tune_canny(threshold1=100, threshold2=200, kernel=5):
    k = kernel if kernel % 2 == 1 else kernel + 1
    blurred = cv2.GaussianBlur(gray, (k, k), 0)
    edges = cv2.Canny(blurred, threshold1, threshold2)
    show([('Исходное изображение', img),
          (f'Canny: {threshold1}, {threshold2}, blur {k}', edges)], cols=2, figsize=(14, 7))


interact(tune_canny,
         threshold1=IntSlider(min=0, max=255, step=5, value=100),
         threshold2=IntSlider(min=0, max=500, step=5, value=200),
         kernel=IntSlider(min=1, max=15, step=2, value=5));

In [ ]:
objects = load_image()
gray_obj = cv2.cvtColor(objects, cv2.COLOR_BGR2GRAY)
thresh = split_touching(objects, binarize_uneven(gray_obj))

contours_all, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
contours_ext, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

min_area = 0.001 * gray_obj.shape[0] * gray_obj.shape[1]
contours_ext = [c for c in contours_ext if cv2.contourArea(c) > min_area]

all_img = objects.copy()
cv2.drawContours(all_img, contours_all, -1, (0, 255, 0), 2)

ext_img = objects.copy()
cv2.drawContours(ext_img, contours_ext, -1, (0, 0, 255), 2)

rect_img = objects.copy()
for c in contours_ext:
    x, y, w, h = cv2.boundingRect(c)
    cv2.rectangle(rect_img, (x, y), (x + w, y + h), (255, 0, 0), 2)

show([('Бинарное изображение', thresh),
      ('Все контуры', all_img),
      ('Внешние контуры', ext_img),
      ('Ограничивающие прямоугольники', rect_img)])

print('Количество предметов:', len(contours_ext))
for i, c in enumerate(contours_ext, 1):
    print(f'Контур {i}: длина = {cv2.arcLength(c, True):.2f}, площадь = {cv2.contourArea(c):.2f}')

In [ ]:
os.makedirs('objects', exist_ok=True)
for i, c in enumerate(contours_ext, 1):
    x, y, w, h = cv2.boundingRect(c)
    cv2.imwrite(f'objects/object_{i}.png', objects[y:y + h, x:x + w])

shutil.make_archive('objects', 'zip', 'objects')
files.download('objects.zip')

In [ ]:
circles_img = load_image()
gray_c = cv2.cvtColor(circles_img, cv2.COLOR_BGR2GRAY)
gray_c = cv2.medianBlur(gray_c, 5)
m = min(gray_c.shape)


def detect_circles(dp=1.2, minDist=40, param1=100, param2=30, minRadius=15, maxRadius=40):
    circles = cv2.HoughCircles(gray_c, cv2.HOUGH_GRADIENT, dp, minDist,
                               param1=param1, param2=param2,
                               minRadius=minRadius, maxRadius=maxRadius)
    out = circles_img.copy()
    count = 0
    if circles is not None:
        circles = np.around(circles).astype(int)
        count = circles.shape[1]
        for x, y, r in circles[0]:
            cv2.circle(out, (int(x), int(y)), int(r), (0, 255, 0), 2)
            cv2.circle(out, (int(x), int(y)), 2, (0, 0, 255), 3)
    show([(f'Найдено окружностей: {count}', out)], cols=1, figsize=(8, 8))
    print('Количество окружностей:', count)


detect_circles()

In [ ]:
interact(detect_circles,
         dp=FloatSlider(min=1.0, max=2.5, step=0.1, value=1.2),
         minDist=IntSlider(min=5, max=m, step=1, value=40),
         param1=IntSlider(min=20, max=300, step=10, value=100),
         param2=IntSlider(min=10, max=100, step=1, value=30),
         minRadius=IntSlider(min=0, max=m, step=1, value=15),
         maxRadius=IntSlider(min=0, max=m, step=1, value=40));

In [ ]:
shapes = load_image()
gray_s = cv2.cvtColor(shapes, cv2.COLOR_BGR2GRAY)
thresh_s = binarize(gray_s)

contours_s, _ = cv2.findContours(thresh_s, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
min_area_s = 0.001 * gray_s.shape[0] * gray_s.shape[1]
contours_s = [c for c in contours_s if cv2.contourArea(c) > min_area_s]

target_vertices = 4
names = {3: 'треугольников', 4: 'прямоугольников'}
counts = {3: 0, 4: 0}
out = shapes.copy()

for c in contours_s:
    perimeter = cv2.arcLength(c, True)
    approx = cv2.approxPolyDP(c, 0.04 * perimeter, True)
    vertices = len(approx)
    if vertices in counts:
        counts[vertices] += 1
    if vertices == target_vertices:
        cv2.drawContours(out, [approx], -1, (0, 255, 0), 3)

show([('Бинарное изображение', thresh_s),
      (f'Найдено {names[target_vertices]}: {counts[target_vertices]}', out)], cols=2, figsize=(14, 7))

print(f'Количество {names[target_vertices]}:', counts[target_vertices])